# Aula 7 — Fundamentos de Redes Neurais

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

O Keras já vem instalado no Google Colab. Nenhuma instalação é
necessária.

## Parte A: Demonstração

### Os dados: 600 torras de café

Cada linha é uma torra, com a temperatura do tambor, o tempo, e se o café
saiu bom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

# Endereço dos dados desta aula no GitHub.
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/torra_cafe.csv"
# Alternativa para testar offline:
# URL_DADOS = "../../data/torra_cafe.csv"

dados = pd.read_csv(URL_DADOS)
print(f"{len(dados)} torras, {dados['boa'].mean():.1%} delas boas")
dados.head()

In [ ]:
boas = dados[dados["boa"] == 1]
ruins = dados[dados["boa"] == 0]

plt.figure(figsize=(8, 5))
plt.scatter(ruins["temperatura_c"], ruins["tempo_min"], label="Torra ruim")
plt.scatter(boas["temperatura_c"], boas["tempo_min"], marker="^", label="Torra boa")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Tempo (min)")
plt.title("As torras boas formam uma ilha")
plt.legend()
plt.show()

### Primeiro, a régua: a regressão logística

É o modelo da Aula 3. A fronteira que ele sabe desenhar é uma reta.

In [ ]:
from sklearn.linear_model import LogisticRegression

entradas = dados[["temperatura_c", "tempo_min"]]
alvo = dados["boa"]

logistica = LogisticRegression(max_iter=1000)
logistica.fit(entradas, alvo)

print(f"Acurácia da logística: {logistica.score(entradas, alvo):.3f}")
print(f"Chutar 'ruim' para todo mundo: {1 - alvo.mean():.3f}")

### Normalizar antes de treinar a rede

A temperatura vai a 235 e o tempo a 17: uma escala é catorze vezes a
outra. Toda rede neural precisa das colunas na mesma escala:

$$x_{\text{normalizado}} = \frac{x - \text{média}}{\text{desvio padrão}}$$

In [ ]:
media = entradas.mean()
desvio = entradas.std()
entradas_norm = (entradas - media) / desvio

print(entradas_norm.describe().round(2))

### A rede, em cinco linhas

Cada neurônio faz a soma de sempre e passa por uma dobra:

$$a = f(w_0 + w_1 x_1 + w_2 x_2) \qquad \text{ReLU}(z) = \max(0, z)$$

In [ ]:
keras.utils.set_random_seed(42)

rede = keras.Sequential([
    keras.layers.Input(shape=(2,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])

rede.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rede.summary()

### Treinar

Uma época é uma passada por todas as torras. Com lote de 32 e 600 torras,
são 19 ajustes de pesos por época.

In [ ]:
historico = rede.fit(entradas_norm, alvo, epochs=120, batch_size=32,
                     validation_split=0.25, verbose=0)

perda, acuracia = rede.evaluate(entradas_norm, alvo, verbose=0)
print(f"Acurácia da rede: {acuracia:.3f}")
print(f"Acurácia da logística: {logistica.score(entradas, alvo):.3f}")

In [ ]:
# A curva de treino: o painel de instrumentos da rede
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(historico.history["loss"], label="treino")
plt.plot(historico.history["val_loss"], label="validação")
plt.xlabel("Época")
plt.ylabel("Perda")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(historico.history["accuracy"], label="treino")
plt.plot(historico.history["val_accuracy"], label="validação")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()

plt.show()

### A fronteira que a rede aprendeu

Perguntamos ao modelo o que ele acha de cada ponto de uma grade, e
pintamos o resultado.

In [ ]:
def desenhar_fronteira(prever, titulo):
    # Monta uma grade de temperaturas e tempos e pergunta a decisão de cada ponto
    grade_temperatura = np.linspace(178, 237, 200)
    grade_tempo = np.linspace(5.5, 17.5, 200)
    T, M = np.meshgrid(grade_temperatura, grade_tempo)
    pontos = np.column_stack([T.ravel(), M.ravel()])
    Z = prever(pontos).reshape(T.shape)

    plt.figure(figsize=(8, 5))
    plt.contourf(T, M, Z, levels=[0, 0.5, 1], alpha=0.3)
    plt.scatter(ruins["temperatura_c"], ruins["tempo_min"], label="Torra ruim")
    plt.scatter(boas["temperatura_c"], boas["tempo_min"], marker="^", label="Torra boa")
    plt.xlabel("Temperatura (°C)")
    plt.ylabel("Tempo (min)")
    plt.title(titulo)
    plt.legend()
    plt.show()

def prever_com_a_rede(pontos):
    tabela = pd.DataFrame(pontos, columns=["temperatura_c", "tempo_min"])
    normalizados = (tabela - media) / desvio
    return (rede.predict(normalizados, verbose=0)[:, 0] >= 0.5).astype(int)

desenhar_fronteira(lambda pontos: logistica.predict(pontos), "A reta da regressão logística")
desenhar_fronteira(prever_com_a_rede, "A ilha que a rede fechou")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo as torras

Rode a célula e observe: qual é a temperatura média das torras boas? E
das ruins?

In [ ]:
print(dados.groupby("boa")[["temperatura_c", "tempo_min"]].mean().round(1))

In [ ]:
if len(dados) == 600:
    print(f"✅ São {len(dados)} torras, como esperado.")
else:
    print("❌ Confira se você rodou a célula que carrega os dados.")

### Exercício 2: normalizando

Crie `minhas_entradas_norm`, com as duas colunas na mesma escala:

$$x_{\text{normalizado}} = \frac{x - \text{média}}{\text{desvio padrão}}$$

In [ ]:
minhas_entradas = dados[["temperatura_c", "tempo_min"]]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(minhas_entradas_norm.mean().round(3))
print(minhas_entradas_norm.std().round(3))

In [ ]:
if abs(minhas_entradas_norm.mean().sum()) < 0.01 and abs(minhas_entradas_norm.std().sum() - 2) < 0.01:
    print("✅ Média perto de zero e desvio perto de 1 nas duas colunas.")
else:
    print("❌ Confira: subtraia a média e divida pelo desvio padrão de cada coluna.")

### Exercício 3: montando a sua rede

Monte uma rede chamada `minha_rede` com **uma** camada escondida de 8
neurônios com ReLU, e a saída com sigmoide. Depois compile com
`optimizer="adam"` e `loss="binary_crossentropy"`.

In [ ]:
keras.utils.set_random_seed(7)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_rede.summary()

In [ ]:
if minha_rede.count_params() == 33:
    print("✅ 33 pesos: 2×8+8 na camada escondida, e 8×1+1 na saída.")
else:
    print(f"❌ Esperava 33 pesos e vieram {minha_rede.count_params()}. Confira o número de neurônios.")

### Exercício 4: treinando

Treine `minha_rede` por 120 épocas, com lote de 32 e `validation_split=0.25`.
Guarde o resultado em `meu_historico`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_perda, minha_acuracia = minha_rede.evaluate(minhas_entradas_norm, alvo, verbose=0)
print(f"Acurácia: {minha_acuracia:.3f}")

In [ ]:
if minha_acuracia > 0.82:
    print("✅ Bem acima dos 71,8% de chutar 'ruim' para todo mundo.")
else:
    print("❌ Acurácia baixa. Confira se você treinou com as entradas normalizadas.")

### Exercício 5: lendo a curva de treino

Desenhe a perda de treino e a de validação, e responda: as duas caem
juntas?

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(meu_historico.history["loss"], label="treino")
plt.plot(meu_historico.history["val_loss"], label="validação")
plt.xlabel("Época")
plt.ylabel("Perda")
plt.legend()
plt.show()

In [ ]:
print("Converse com um colega: se a linha de validação subisse enquanto a de treino cai, o que estaria acontecendo?")

### Exercício 6: mais neurônios ajudam?

Treine uma rede com **32** neurônios na camada escondida e compare a
acurácia com a de 8.

In [ ]:
keras.utils.set_random_seed(7)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
acuracia_grande = rede_grande.evaluate(minhas_entradas_norm, alvo, verbose=0)[1]
print(f"Rede de 8 neurônios:  {minha_acuracia:.3f}")
print(f"Rede de 32 neurônios: {acuracia_grande:.3f}")

In [ ]:
print("Compare os dois: com 8 dobras a ilha fica cercada por um polígono grosseiro.")
print("Com 32, o contorno acompanha melhor a borda. Mais dobras, mais formato.")

### Exercício 7: desafio, uma torra nova

O mestre quer testar 208 °C por 12 minutos. Lembre de normalizar antes de
perguntar ao modelo, com a **mesma** média e o **mesmo** desvio do treino.

In [ ]:
torra_nova = pd.DataFrame({"temperatura_c": [208.0], "tempo_min": [12.0]})

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Probabilidade de sair boa: {probabilidade:.0%}")

In [ ]:
if probabilidade > 0.5:
    print("✅ O modelo aposta que essa torra sai boa. Ela está bem no meio da ilha.")
else:
    print("❌ Confira se você usou a média e o desvio do conjunto de treino.")

Agora, em texto: explique em duas ou três frases por que a regressão
logística não conseguiu resolver este problema, e o que a rede tem que
ela não tem. Edite esta célula (duplo clique nela) e escreva sua resposta
no lugar deste parágrafo.